# Notebook 1 — Load and Clean Data

**Aim:** load the six years (2020-2025) of raw MOT test extracts and turn them into one trustworthy, deduplicated table of MOT tests.

**Reads:** the raw CSV files in `dft_test_result_extracts_2020/` … `dft_test_result_extracts_2025/`.

**Produces:**
- `processed_data/cleaned_mot_tests/` — the cleaned MOT test records, saved as one Parquet file per year so this notebook never has to hold all six years in memory at once. Later notebooks read the whole folder with `pd.read_parquet(...)` and it behaves like a single table.
- `processed_data/quality_summary_stage1.json` — a small summary of record counts and data-quality issues, so Notebook 7 (conclusions and limitations) can report on data quality without reloading the raw data.

The raw extracts are large (around 22 GB in total across six years), so this notebook processes **one year at a time**: load that year's twelve monthly files, clean them, save them, and free the memory before moving to the next year. This keeps the notebook usable on an ordinary laptop.

This notebook only does *row-level* cleaning (missing values, duplicates, negative mileage) plus one simple cross-vehicle check (inconsistent make/model/fuel type/postcode area). Checks that depend on comparing a vehicle's *consecutive* tests — decreasing odometer readings, unusually short or long intervals, implausibly high annualised mileage — need each vehicle's full test history assembled in order, which is what Notebook 2 (vehicle intervals) does.

In [20]:
import csv
import json
import sys
from pathlib import Path

import pandas as pd

# The shared configuration now lives in the config package beside this notebook.
for candidate_dir in [Path.cwd(), Path.cwd() / 'mileage_prediction']:
    if (candidate_dir / 'config' / '__init__.py').exists():
        sys.path.insert(0, str(candidate_dir))
        break

import config

print(f'Project root: {config.PROJECT_ROOT}')
print(f'Processed data folder: {config.PROCESSED_DATA_DIR}')
print(f'Years to load: {config.YEARS_TO_LOAD}')
print(f'Required columns: {config.REQUIRED_COLUMNS}')

Project root: /Users/matthewsalter/Documents/Development/eVED work/data-science
Processed data folder: /Users/matthewsalter/Documents/Development/eVED work/data-science/processed_data
Years to load: [2020, 2021, 2022, 2023, 2024, 2025]
Required columns: ['test_id', 'vehicle_id', 'test_date', 'test_mileage', 'make', 'model', 'fuel_type', 'postcode_area', 'first_use_date', 'test_type', 'test_result', 'test_class_id', 'completed_date']


## Step 1: find the raw files

Check that every year's folder exists and contains monthly CSV files before doing any work. This gives a clear error message straight away if the notebook is run from the wrong working directory, rather than failing partway through.

In [12]:
year_to_files = {}
missing_paths = []

for year in config.YEARS_TO_LOAD:
    year_dir = config.RAW_DATA_DIRS[year]
    if not year_dir.exists():
        missing_paths.append(str(year_dir))
        continue
    monthly_files = sorted(year_dir.glob('dft_test_result_extract_*.csv'))
    if not monthly_files:
        missing_paths.append(f'{year_dir} (folder exists but no CSV files were found in it)')
    year_to_files[year] = monthly_files

if missing_paths:
    raise FileNotFoundError(
        'Could not find the expected raw data. Run this notebook with the project '
        'directory as the working directory, or check these paths:\n'
        + '\n'.join(missing_paths)
    )

for year, files in year_to_files.items():
    print(f'{year}: {len(files)} monthly files found')

2020: 12 monthly files found
2021: 12 monthly files found
2022: 12 monthly files found
2023: 12 monthly files found
2024: 12 monthly files found
2025: 12 monthly files found


## Step 2: define the per-year cleaning steps

For each year, this:

1. Loads the twelve monthly files and stacks them into one table, recording `source_file` and `source_year` for traceability.
2. Strips whitespace from text fields and turns blank or placeholder text (like `""` or `"nan"`) into a proper missing value.
3. Removes rows missing an essential field (`test_id`, `vehicle_id`, `test_date`, or `test_mileage`) — these rows can't be used for anything downstream.
4. Removes rows with negative mileage, which are always data errors.
5. Removes exact duplicate `test_id` records, keeping the first occurrence.
6. Removes repeated vehicle/date/mileage records — the same vehicle, test date, and mileage appearing more than once, which usually means the same test was captured twice.

At every step, the number of rows removed is recorded so nothing is silently discarded.

In [13]:
# Numeric columns are given a compact dtype at load time, rather than parsed afterwards,
# to keep peak memory down while a whole year's extracts are held in memory.
NUMERIC_DTYPES = {
    'test_id': 'int64',
    'vehicle_id': 'int64',
    'test_mileage': 'float32',
    'test_class_id': 'Int16',
}
DATE_COLUMNS = ['test_date', 'first_use_date']
TEXT_COLUMNS = ['make', 'model', 'fuel_type', 'postcode_area', 'test_type', 'test_result']
ESSENTIAL_COLUMNS = ['test_id', 'vehicle_id', 'test_date', 'test_mileage']


def load_one_year(year, file_paths):
    """Read every monthly extract for one year into a single DataFrame."""
    parts = []
    for file_path in file_paths:
        part = pd.read_csv(
            file_path,
            usecols=config.REQUIRED_COLUMNS,
            dtype=NUMERIC_DTYPES,
            parse_dates=DATE_COLUMNS,
            # A handful of rows contain stray " characters (e.g. in model/colour text);
            # QUOTE_NONE treats " as an ordinary character instead of a field delimiter.
            quoting=csv.QUOTE_NONE,
            # low_memory=False avoids a pandas C-parser bug where chunked reads combined
            # with QUOTE_NONE can raise an internal IndexError while merging chunks.
            low_memory=False,
        )
        part['source_file'] = file_path.name
        part['source_year'] = year
        parts.append(part)
    return pd.concat(parts, ignore_index=True)


def standardise_text(dataframe, columns):
    """Strip whitespace and turn blank or placeholder text into a missing value."""
    for column in columns:
        cleaned = dataframe[column].astype('string').str.strip()
        cleaned = cleaned.replace({'': pd.NA, 'nan': pd.NA, 'NaN': pd.NA, 'None': pd.NA})
        dataframe[column] = cleaned
    return dataframe


def clean_one_year(year, file_paths):
    """Load, clean, and count quality issues for one year. Returns (cleaned_df, counts)."""
    raw = load_one_year(year, file_paths)
    raw = standardise_text(raw, TEXT_COLUMNS)

    counts = {'raw_records': len(raw)}

    missing_essential = raw[ESSENTIAL_COLUMNS].isna().any(axis=1)
    counts['missing_essential_fields'] = int(missing_essential.sum())
    cleaned = raw.loc[~missing_essential].copy()

    negative_mileage = cleaned['test_mileage'] < 0
    counts['negative_mileage'] = int(negative_mileage.sum())
    cleaned = cleaned.loc[~negative_mileage].copy()

    duplicate_test_id = cleaned.duplicated(subset='test_id', keep='first')
    counts['duplicate_test_id_records'] = int(duplicate_test_id.sum())
    cleaned = cleaned.loc[~duplicate_test_id].copy()

    repeated_vehicle_date_mileage = cleaned.duplicated(
        subset=['vehicle_id', 'test_date', 'test_mileage'], keep='first'
    )
    counts['repeated_vehicle_date_mileage_records'] = int(repeated_vehicle_date_mileage.sum())
    cleaned = cleaned.loc[~repeated_vehicle_date_mileage].copy()

    counts['usable_records'] = len(cleaned)
    return cleaned, counts

## Step 3: run the cleaning for every year and save the results

Process one year at a time: clean it, write it straight to disk, and record its attribute combinations (vehicle_id plus make/model/fuel_type/postcode_area) for the cross-vehicle check in Step 4, then discard the full year's data before moving on. This is what keeps peak memory bounded to roughly one year's worth of data rather than all six.

In [14]:
ATTRIBUTE_COLUMNS = ['vehicle_id', 'make', 'model', 'fuel_type', 'postcode_area']

config.ensure_processed_data_dir()
cleaned_output_dir = config.PROCESSED_DATA_DIR / 'cleaned_mot_tests'
cleaned_output_dir.mkdir(parents=True, exist_ok=True)

quality_summary = {'years': {}}
vehicle_attribute_frames = []

for year in config.YEARS_TO_LOAD:
    print(f'Cleaning {year}: {len(year_to_files[year])} monthly files...')
    cleaned_year, counts = clean_one_year(year, year_to_files[year])
    quality_summary['years'][str(year)] = counts

    year_output_path = cleaned_output_dir / f'source_year={year}.parquet'
    cleaned_year.to_parquet(year_output_path, index=False)

    vehicle_attribute_frames.append(cleaned_year[ATTRIBUTE_COLUMNS].drop_duplicates())

    print(f'  Raw: {counts["raw_records"]:,}  Usable: {counts["usable_records"]:,}')
    del cleaned_year

print('\nAll years cleaned and saved to', cleaned_output_dir)

Cleaning 2020: 12 monthly files...
  Raw: 38,594,013  Usable: 35,555,875
Cleaning 2021: 12 monthly files...
  Raw: 40,380,646  Usable: 38,019,313
Cleaning 2022: 12 monthly files...
  Raw: 41,632,878  Usable: 39,257,259
Cleaning 2023: 12 monthly files...
  Raw: 42,216,721  Usable: 39,870,684
Cleaning 2024: 12 monthly files...
  Raw: 42,637,055  Usable: 40,572,652
Cleaning 2025: 12 monthly files...
  Raw: 42,728,066  Usable: 40,673,014

All years cleaned and saved to /Users/matthewsalter/Documents/Development/eVED work/data-science/processed_data/cleaned_mot_tests


In [24]:
# Optional Step 3b: rebuild the attribute frames from existing cleaned Parquet files.
# Run this cell instead of rerunning Step 3 when the cleaned outputs already exist.
ATTRIBUTE_COLUMNS = ['vehicle_id', 'make', 'model', 'fuel_type']
cleaned_output_dir = config.PROCESSED_DATA_DIR / 'cleaned_mot_tests'
parquet_files = sorted(cleaned_output_dir.glob('source_year=*.parquet'))

if not parquet_files:
    raise FileNotFoundError(
        f'No cleaned Parquet files found in {cleaned_output_dir}. '
        'Run Step 3 first.'
    )

vehicle_attribute_frames = [
    pd.read_parquet(path, columns=ATTRIBUTE_COLUMNS).drop_duplicates()
    for path in parquet_files
]

print(f'Rebuilt {len(vehicle_attribute_frames)} vehicle attribute frames from saved Parquet files.')

Rebuilt 6 vehicle attribute frames from saved Parquet files.


## Step 4: check for inconsistent vehicle attributes across tests

The same `vehicle_id` should have the same make, model at every test. If it doesn't, that points to either a data-entry error or a `vehicle_id` being reused for more than one physical vehicle. This is reported as a count for now — it isn't used to filter any records here.

In [25]:
vehicle_attributes = pd.concat(vehicle_attribute_frames, ignore_index=True).drop_duplicates()
del vehicle_attribute_frames

attribute_nunique_per_vehicle = vehicle_attributes.groupby('vehicle_id').nunique()
inconsistent_vehicles = (
    attribute_nunique_per_vehicle[['make', 'model']] > 1
).any(axis=1)

vehicles_checked = int(attribute_nunique_per_vehicle.shape[0])
vehicles_with_inconsistent_attributes = int(inconsistent_vehicles.sum())

quality_summary['inconsistent_vehicle_attributes'] = {
    'vehicles_checked': vehicles_checked,
    'vehicles_with_inconsistent_attributes': vehicles_with_inconsistent_attributes,
}

print(
    f'{vehicles_with_inconsistent_attributes:,} of {vehicles_checked:,} vehicles '
    'have more than one value for make or model'
    f'({vehicles_with_inconsistent_attributes / vehicles_checked:.2%}).'
)

677,182 of 44,422,222 vehicles have more than one value for make or model(1.52%).


## Step 5: save the data-quality summary

Bring the per-year counts together with overall totals and save them as JSON. Notebook 7 reads this file directly instead of reloading any raw data.

In [26]:
quality_summary['totals'] = {
    'source_files_loaded': sum(len(files) for files in year_to_files.values()),
    'raw_records_loaded': sum(c['raw_records'] for c in quality_summary['years'].values()),
    'usable_tests': sum(c['usable_records'] for c in quality_summary['years'].values()),
}

summary_path = config.PROCESSED_DATA_DIR / 'quality_summary_stage1.json'
with open(summary_path, 'w') as summary_file:
    json.dump(quality_summary, summary_file, indent=2)

print(f'Saved quality summary to {summary_path}')
quality_summary['totals']

Saved quality summary to /Users/matthewsalter/Documents/Development/eVED work/data-science/processed_data/quality_summary_stage1.json


{'source_files_loaded': 72,
 'raw_records_loaded': 248189379,
 'usable_tests': 233948797}

## Step 6: sanity-check the saved output

Reload the cleaned data straight from the Parquet folder, exactly as Notebook 2 will, to confirm it looks right before moving on.

In [27]:
check_df = pd.read_parquet(cleaned_output_dir)
print(f'Shape: {check_df.shape}')
print(check_df.dtypes)
check_df.head()

Shape: (233948797, 15)
test_id                    int64
vehicle_id                 int64
test_date         datetime64[us]
test_class_id              Int16
test_type                 string
test_result               string
test_mileage             float32
postcode_area             string
make                      string
model                     string
fuel_type                 string
first_use_date               str
completed_date               str
source_file                  str
source_year                int64
dtype: object


,test_id,vehicle_id,test_date,test_class_id,test_type,test_result,test_mileage,postcode_area,make,model,fuel_type,first_use_date,completed_date,source_file,source_year
0,666422869,1253657552,2020-01-01,4,NT,P,63975.0,TR,CITROEN,DISPATCH,DI,2011-03-14,NaN,dft_test_result_extract_202001.csv,2020
1,623774383,51021182,2020-01-01,4,NT,P,107361.0,NN,SEAT,IBIZA,PE,2008-12-18,NaN,dft_test_result_extract_202001.csv,2020
2,581125897,612989654,2020-01-01,4,NT,P,73160.0,NN,MERCEDES,A 150,PE,2007-09-28,NaN,dft_test_result_extract_202001.csv,2020
3,325234981,1422080365,2020-01-01,1,NT,F,27120.0,SS,KTM,125,PE,2013-12-07,NaN,dft_test_result_extract_202001.csv,2020
4,367883467,1254023710,2020-01-01,4,NT,P,81260.0,RM,FORD,FOCUS,PE,2005-06-13,NaN,dft_test_result_extract_202001.csv,2020


## Recap and next step

This notebook has produced:

- `processed_data/cleaned_mot_tests/` — one cleaned, deduplicated, traceable Parquet file per year.
- `processed_data/quality_summary_stage1.json` — raw/usable record counts and the counts excluded for each cleaning rule.

Next: **Notebook 2 (`02_vehicle_intervals.ipynb`)** reads this cleaned table, reconstructs each vehicle's test history, and turns consecutive tests into annualised mileage intervals — including the decreasing-odometer, interval-length, and implausible-mileage checks that need a vehicle's full history to compute.